# Preamble — load data + Figure-4 GAM export

In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src.notebook import load_notebook_data

In [ ]:
dataset       = 'HCP-D' # 'HCP-D' 'PNC'
which_modality = 'FMRI'
n_regions = 400

if dataset == 'HCP-D':
    output_type   = 'timescale_auc_zero_crossing_bp-0.01-0.08hz'
elif dataset == 'PNC':
    output_type   = 'timescale_auc_zero_crossing'
    # output_type   = 'timescale_zero_crossing'
    # output_type   = 'timescale_zero_crossing_robust_nc2'
    # output_type   = 'timescale_zero_crossing_robust_nc3'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   output_type=output_type, use_resids=True,
                                   load_subject_data=True, n_regions=n_regions)

timescales_processor = notebook_data['timescales_processor']
data_dict            = notebook_data['data_dict']
results_dir          = notebook_data['results_dir']
df_neuro             = notebook_data['supporting_data']['df_neuro']

sa_axis = df_neuro['SA_Axis'].values

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
import os
import numpy as np

# Load the cross-notebook GAM + regional export written by timescales_analysis_fig4.ipynb
# (cell: 'Save regional and decile-level GAM outputs ...'). This decouples Figure 5 from the
# upstream INT/GAM pipeline — no refitting here.
gam_out_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'gam')
gam_file    = os.path.join(gam_out_dir, f'{dataset}_{n_regions}_regional_gam_results.npy')
if not os.path.exists(gam_file):
    raise FileNotFoundError(
        f'GAM export not found:\n  {gam_file}\n'
        'Run timescales_analysis_fig4.ipynb (through its "Save regional and decile-level GAM '
        'outputs" cell) first to generate it.')
_d = np.load(gam_file, allow_pickle=True).item()

# The standalone-notebook keys (rest_reg / defl_reg / df_behavior / sa_axis) were added to the
# fig4 save cell when Figure 5 was split out. An older export predating that change will be
# missing them — re-run the fig4 save cell to regenerate.
_required = ['age_grid', 'rest_reg', 'defl_reg', 'df_behavior', 'sa_axis',
             'rest_pred', 'rest_deriv', 'task_pred', 'task_deriv',
             'deflection_pred', 'deflection_deriv']
_missing = [k for k in _required if k not in _d]
if _missing:
    raise KeyError(
        f'GAM export {os.path.basename(gam_file)} is missing keys {_missing}. '
        'It predates the Figure-5 split — re-run the fig4 "Save regional ... GAM outputs" cell '
        '(now extended to also save rest_reg / defl_reg / df_behavior / sa_axis).')

age_grid    = _d['age_grid']
rest_reg    = _d['rest_reg']
defl_reg    = _d['defl_reg']
df_behavior = _d['df_behavior']
sa_axis     = _d['sa_axis']          # exact axis used when the stage maps were fit

# Reconstruct the GAM smooth dict in the shape the Figure-5 cells expect.
results_reg = {sig: {'pred': _d[f'{sig}_pred'], 'deriv': _d[f'{sig}_deriv']}
               for sig in ('rest', 'task', 'deflection')}

print(f'Loaded GAM export <- {gam_file}')
print(f'  results_reg signals: {list(results_reg)} | age_grid {age_grid.shape}')
print(f'  rest_reg {rest_reg.shape} | defl_reg {defl_reg.shape} | df_behavior {df_behavior.shape}')


# Figure 5 — developmental-stage analyses of task-evoked INT modulation

Two developmental **stage maps** of net change in task-evoked |ΔINT|, from the per-parcel
Schaefer-400 GAMs fit in `timescales_analysis_fig4.ipynb` (**stage 1**: age_min → inflection;
**stage 2**: inflection → age_max). Flow:

1. compute the stage maps and plot their surfaces;
2. one **collinearity check** of both stage maps against the candidate covariates
   (S-A axis, baseline-INT, baseline-|ΔINT|);
3. **choose one covariate control per stage** and compute gene + PET spatial correlations
   (BrainSMASH two-tailed null, BH-FDR within family);
4. Yeo-7 network enrichment;
5. an exploratory stage-2 DMN-vs-behaviour test (computed, not plotted);
6. assemble Figure 5.

No covariate ladder — a single, explicitly chosen model per stage.

In [ ]:
# ── Spatial-null configuration for this section ───────────────────────────────
# Number of BrainSMASH surrogates used for every spatial-null p-value below
# (PET markers, AHBA genes, Yeo-7 enrichment). Set low (500) for fast testing;
# raise to 5000 for publication-quality nulls. Surrogates are cached per
# (map, n_perms), so changing this value regenerates the cache at the new count.
N_PERMS = 5000

# Stage-1/stage-2 boundary age (years) for compute_stage_maps below. Previously defined
# in the (now-removed) SES section; kept here so this section is self-contained.
inflection_age = 17

# ── Family-map inclusion toggles ──────────────────────────────────────────────
# Selectively exclude an entire annotation-map family from the Figure 5 analysis.
# All True by default (every family included). Set a flag False to drop that family:
#   INCLUDE_GENE_MAPS   -> AHBA gene-expression maps (df_genes)
#   INCLUDE_PET_MAPS    -> PET receptor/transporter maps (df_pet)
#   INCLUDE_MYELIN_MAPS -> Baum T1w/T2w myelin maps (df_baum)
# An excluded family's DataFrame is set to None, so all downstream spatial-correlation,
# figure, and per-metric cells skip it via their existing `is None` guards.
INCLUDE_GENE_MAPS   = True
INCLUDE_PET_MAPS    = False
INCLUDE_MYELIN_MAPS = True


In [ ]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from tqdm import tqdm

# Spatial null models now live in src.nulls (single source of truth; see that module's docstring).
# The distance basis is per-hemisphere GEODESIC distance along the fsLR cortical surface -- the
# metric BrainSMASH recommends for surface parcellations -- replacing the old Euclidean centroid
# basis. load_distance_matrix builds+caches the geodesic matrix (from surfaces/dlabels bundled in
# snaplab_tools/nulls) on first use.
from src import nulls
from src.nulls import corr_with_covar_null

# Cache dir for surrogates. Cache filenames embed the distance `kind`, so old Euclidean nulls on
# disk are NOT reused by the geodesic runs (they are recomputed and cached separately).
surrogate_cache_dir = nulls.surrogate_cache_dir(timescales_processor, dataset)

# Per-hemisphere geodesic distance basis + hemisphere labels (Schaefer 7-network order: the
# within-hemisphere blocks hold geodesic distances; the cross-hemisphere block is NaN).
distance_matrix, hemi_labels = nulls.load_distance_matrix(n_regions, kind='geodesic')


def generate_surrogates(brain_map, distance_matrix=distance_matrix, name=None,
                        cache_dir=surrogate_cache_dir, n_regions=n_regions, n_perms=5000):
    """Notebook adapter over src.nulls.generate_surrogates: per-hemisphere geodesic BrainSMASH
    surrogates with NaN handling and on-disk caching. Returns (n_perms, n_regions)."""
    return nulls.generate_surrogates(
        brain_map, name=name, n_regions=n_regions, n_perms=n_perms, kind='geodesic',
        distance_matrix=distance_matrix, hemi=hemi_labels, cache_dir=cache_dir)


def correlate_family(brain_map, df_targets, covariates, distance_matrix=distance_matrix,
                     n_perms=5000, desc='targets'):
    """Notebook adapter over src.nulls.correlate_family (per-hemisphere geodesic basis); surrogates
    cached per target map, then BH-FDR within the family."""
    return nulls.correlate_family(
        brain_map, df_targets, covariates=covariates, n_regions=n_regions, kind='geodesic',
        n_perms=n_perms, cache_dir=surrogate_cache_dir, desc=desc,
        distance_matrix=distance_matrix, hemi=hemi_labels)


In [ ]:
from src.gams import compute_stage_maps

stage1_map, stage2_map = compute_stage_maps(results_reg['deflection'], age_grid,
                                            inflection_age=inflection_age, annualized=True)

## Stage-map surfaces

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from snaplab_tools.plotting.plotting import plot_brain_surface_data

# Stage-map surfaces, rendered as soon as the maps exist. Saved as standalone figures on a shared
# symmetric diverging scale (for manual assembly into the final figure) and shown inline here.

if n_regions == 400:
    surface_paths = {}
    for _lab, _map in [('stage1', stage1_map), ('stage2', stage2_map)]:
        _vmax = np.round(float(np.nanmax(np.abs(_map))), 2)
        
        _fig, _ = plot_brain_surface_data(data_vector=_map, parcellation=f'schaefer_{n_regions}-7', cmap='coolwarm', figsize=(1.85, 1.5),
                                        vmin=-_vmax, vmax=_vmax, cbar_label='') # '|INT defl.| annualized change'
        surface_paths[_lab] = os.path.join(nas_scratch_dir, f'figure_5_surface_{_lab}.png')
        _fig.savefig(surface_paths[_lab], dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
    print('Saved stage-map surfaces ->', surface_paths)


## Collinearity / over-control check

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from snaplab_tools.plotting.plotting import plot_correlation

# ── Candidate covariate maps, per stage ───────────────────────────────────────
# Baselines = GAM-fitted maps at each stage's START age (stage 1: youngest; stage 2: inflection),
# reconstructed on the original spatial scale (GAMs fit demeaned -> add back each region's
# subject-mean). S-A axis is shared. These three are the only covariates considered below.
_cohort_rest = np.nanmean(rest_reg, axis=0)
_cohort_defl = np.nanmean(defl_reg, axis=0)
_idx_infl = int(np.argmin(np.abs(age_grid - inflection_age)))

covariate_maps = {
    'stage1': {'S-A': sa_axis,
               'baseline-INT':  results_reg['rest']['pred'][:, 0] + _cohort_rest,
               'baseline-dINT': results_reg['deflection']['pred'][:, 0] + _cohort_defl},
    'stage2': {'S-A': sa_axis,
               'baseline-INT':  results_reg['rest']['pred'][:, _idx_infl] + _cohort_rest,
               'baseline-dINT': results_reg['deflection']['pred'][:, _idx_infl] + _cohort_defl},
}
stage_maps = {'stage1': stage1_map, 'stage2': stage2_map}
_cov_names = ['S-A', 'baseline-INT', 'baseline-dINT']
print(f'Stage boundary (inflection) at age_grid[{_idx_infl}] = {age_grid[_idx_infl]:.2f} yr.')

# ── Visual check: each stage map vs each candidate covariate (rows = stage, cols = covariate) ──
fig, axes = plt.subplots(2, 3, figsize=(7, 5), dpi=300)
for _i, _ph in enumerate(['stage1', 'stage2']):
    for _j, _cn in enumerate(_cov_names):
        _ax = axes[_i, _j]
        _x = np.asarray(covariate_maps[_ph][_cn], float)
        _y = np.asarray(stage_maps[_ph], float)
        _m = np.isfinite(_x) & np.isfinite(_y)
        plot_correlation(pd.Series(_x[_m], name=_cn), pd.Series(_y[_m], name=f'{_ph} net Δ|INT|'),
                         _ax, method='pearson', show_stats=True, show_colorbar=False, fontsize=7)
        _ax.tick_params(labelsize=6)
fig.suptitle('Stage maps vs candidate covariates — collinearity / over-control check', fontsize=9)
fig.tight_layout()
plt.show()

# ── Compact numeric summary ───────────────────────────────────────────────────
def _r(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    return sp.stats.pearsonr(a[m], b[m])[0]

collin = pd.DataFrame({ph: {cn: _r(stage_maps[ph], covariate_maps[ph][cn]) for cn in _cov_names}
                       for ph in ['stage1', 'stage2']}).T
pd.set_option('display.float_format', lambda v: f'{v:+.3f}')
print('\nStage map × covariate collinearity (Pearson r):')
print(collin.to_string())
print('\n|r| high (>~0.5) => controlling that covariate in that stage risks over-control '
      '(it absorbs the change-map signal). Use this to set the per-stage covariate choice below.')


## Molecular maps: genes & PET

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
from src.notebook import average_pet_markers

# Molecular reference maps (PET, AHBA genes, cell-types) only exist at Schaefer-400. At any other
# resolution (e.g. n_regions=1000) skip these families outright -- only the Yeo-7 enrichment and the
# myelin section below are run there. Downstream cells guard on `df_pet/df_genes/df_cells is None`.
df_neuro_full   = notebook_data['supporting_data']['df_neuro'].copy()
# distance_matrix now comes from src.nulls (per-hemisphere geodesic basis; set in the setup cell).

if n_regions != 400 or not INCLUDE_PET_MAPS:
    df_pet = None
    individual_cols, composite_cols = [], []
    if n_regions != 400:
        print(f'n_regions={n_regions} != 400; skipping PET family (molecular maps are Schaefer-400 only).')
    else:
        print('INCLUDE_PET_MAPS=False; excluding PET family from Figure 5.')
else:
    # Reuse the exact composite grouping from the existing neurotransmitter analysis
    # (timescales_analysis_fig3.ipynb): 19 individual PET markers, z-scored, then averaged
    # within excitatory / inhibitory / transporter functional categories.
    pet_categories = notebook_data['supporting_data']['pet_categories'].copy()

    pet_category = 'neurotransmitter_function'

    # PET only; z-score each of the 19 individual markers
    neuro_cols = pet_categories.index
    df_pet     = df_neuro_full[neuro_cols].copy()
    for col in neuro_cols:
        df_pet[col] = sp.stats.zscore(df_pet[col])

    # Rename: drop 'PET_' prefix and prepend the functional category (e.g. 'Excitatory_D1')
    neuro_cols = [s.split('_')[1] for s in neuro_cols]
    neuro_cols = ['_'.join(s) for s in zip(*[list(pet_categories[pet_category]), neuro_cols])]
    df_pet.columns       = neuro_cols
    pet_categories.index = neuro_cols

    # Composite maps: z-scored markers averaged within each category -> Mean_Excitatory, etc.
    df_pet_composite = average_pet_markers(df_pet, pet_categories, pet_category)
    df_pet = pd.concat((df_pet, df_pet_composite), axis=1)

    individual_cols = list(pet_categories.index)            # 19 individual markers
    composite_cols  = list(df_pet_composite.columns)        # 3 composites
    print(f'PET maps: {len(individual_cols)} individual markers + {len(composite_cols)} composites '
          f'= {df_pet.shape[1]} total, over {df_pet.shape[0]} parcels')
    print('Composites:', composite_cols)
    print('Individual:', individual_cols)

    # Dominant PET axis (PC1 over the 19 receptor/transporter maps; PET is dense -> full coverage).
    df_pet['PET-PC1'] = df_neuro_full['PCA_PET-PC1'].astype(float).values
    print(f"Added PET-PC1 (receptor PC1); coverage {int(df_pet['PET-PC1'].notna().sum())} parcels.")


In [ ]:
df_neuro_full

In [ ]:
import numpy as np
import pandas as pd

# Curated timescale-related gene set (already present in df_neuro as Gene_*). rest_INT direction
# from the manuscript table is context only (tests are two-tailed).
genes_original = ['GRIN2B', 'GRIN2A', 'GABRA3', 'GABRA1', 'PVALB', 'PRR5', 'KCNA3', 'SCN1A']

# ---- New panel 1: recurrent / glutamatergic synaptic strength (lead hypothesis) ----
genes_recurrent_excitation = [
      'GRIN1', 'GRIN2A', 'GRIN2B',            # NMDA receptor subunits
      'GRIA1', 'GRIA2', 'GRIA3', 'GRIA4',     # AMPA receptor subunits
      'GRM5',                                 # metabotropic glutamate receptor 5
      'CAMK2A', 'CAMK2B',                     # postsynaptic Ca2+/calmodulin kinase
      'DLG4',                                 # PSD-95
      'SYNGAP1', 'SHANK3', 'HOMER1',          # postsynaptic scaffolding / signaling
]

# ---- New panel 2: presynaptic / calcium-regulated exocytosis (Xia/Dong-implicated axis) ----
genes_presynaptic_exocytosis = [
      'SYT1',                                 # synaptotagmin-1 (Ca2+ sensor)
      'STX1A', 'STXBP1', 'SNAP25', 'VAMP2',   # SNARE complex
      'RIMS1',                                # active-zone release machinery
      'CACNA1A', 'CACNA1B',                   # presynaptic Ca2+ channels (P/Q, N-type)
      'SV2A',                                 # synaptic vesicle glycoprotein 2A
      'SYN1', 'SYP',                          # synapsin-1, synaptophysin
      'CPLX1', 'CPLX2',                       # complexins
]

# ---- Combined (deduplicated, order preserved) ----
# gene_curated = list(dict.fromkeys(
#       genes_original + genes_recurrent_excitation + genes_presynaptic_exocytosis
# ))
gene_curated = genes_original

# Gene and cell-type maps only exist at Schaefer-400 (see PET cell above); skip both otherwise.
if n_regions != 400:
    df_genes = None
    df_cells = None
    print(f'n_regions={n_regions} != 400; skipping gene and cell-type families.')
else:
    df_genes = df_neuro_full[[f'Gene_{g}' for g in gene_curated]].copy()
    df_genes.columns = gene_curated
    _n_cov = int(df_genes.iloc[:, 0].notna().sum())
    print(f'AHBA genes: {len(gene_curated)} maps over {df_genes.shape[0]} parcels; '
          f'{_n_cov} with AHBA coverage, {df_genes.shape[0] - _n_cov} NaN (mostly right-hemisphere) '
          f'and dropped per-map by the NaN-masking in generate_surrogates / corr_with_covar_null.')

    # Dominant gene-expression axis (AHBA PC1 over the broad brain-gene set). The stored PC was
    # median-imputed over all parcels, so mask it back to the curated genes' AHBA coverage --
    # it is then tested on the same parcels rather than the imputed (mostly right-hem) ones.
    _gene_pc1 = df_neuro_full['PCA_Gene-PC1'].astype(float).values.copy()
    _gene_pc1[~df_genes.iloc[:, 0].notna().values] = np.nan
    df_genes['Gene-PC1'] = _gene_pc1
    print(f'Added Gene-PC1 (AHBA expression PC1); coverage {int(np.isfinite(_gene_pc1).sum())} parcels.')

    # -- Cell-type maps (Jorstad transcriptomic cell-type expression; 'Cell_' prefix) --
    cell_cols = [c for c in df_neuro_full.columns if c.startswith('Cell_')]
    df_cells  = df_neuro_full[cell_cols].copy()
    df_cells.columns = [c.replace('Cell_', '') for c in cell_cols]
    _n_cov_cell = int(df_cells.iloc[:, 0].notna().sum())
    print(f'Cell-type maps: {len(cell_cols)} maps over {df_cells.shape[0]} parcels; '
          f'{_n_cov_cell} with coverage, {df_cells.shape[0] - _n_cov_cell} NaN (mostly right-hemisphere) '
          f'and dropped per-map by the NaN-masking in generate_surrogates / corr_with_covar_null.')

    # Dominant cell-type axis (PC1 over the cell-type maps); masked back to the cell maps' coverage.
    _cell_pc1 = df_neuro_full['PCA_Cell-PC1'].astype(float).values.copy()
    _cell_pc1[~df_cells.iloc[:, 0].notna().values] = np.nan
    df_cells['Cell-PC1'] = _cell_pc1
    print(f'Added Cell-PC1 (cell-type expression PC1); coverage {int(np.isfinite(_cell_pc1).sum())} parcels.')

# Optional exclusion of the gene family (INCLUDE_GENE_MAPS toggle; default True). Cell-type
# maps are a separate family and remain governed by the n_regions gate above.
if not INCLUDE_GENE_MAPS:
    df_genes = None
    print('INCLUDE_GENE_MAPS=False; excluding gene family from Figure 5.')

# -- Baum myelin maps (precomputed T1w/T2w developmental effects; 'baum-' prefix) --
# Loaded in load_supporting_data (src.notebook._load_baum_myelin_maps). Unlike the molecular
# families above (Schaefer-400 only), the Baum maps are provided at several Schaefer resolutions,
# so this family is NOT gated on n_regions. df_baum columns are the canonical metric names
# (e.g. 'baum-aroc'); these must match the precomputed BrainSMASH null-cache filenames on disk.
df_baum     = notebook_data['supporting_data']['df_baum']
baum_labels = notebook_data['supporting_data']['baum_labels']
if not INCLUDE_MYELIN_MAPS:
    df_baum = None
    print('INCLUDE_MYELIN_MAPS=False; excluding Baum myelin family from Figure 5.')
elif df_baum is not None:
    df_baum = df_baum.copy()
    print(f'Baum myelin maps: {df_baum.shape[1]} maps over {df_baum.shape[0]} parcels '
          f'({", ".join(df_baum.columns)}).')
else:
    print(f'Baum myelin maps not available at n_regions={n_regions}; skipping Baum family.')


## Choose covariates per stage, then compute associations

Set the control **separately for each stage** from the collinearity check, then compute gene + PET partial correlations under that single chosen model (BrainSMASH two-tailed null; BH-FDR within each family). One model per stage — no ladder.

In [ ]:
import numpy as np
import pandas as pd

# ====================== COVARIATE CHOICE (per stage) ==========================
# Each is any subset of ['S-A', 'baseline-INT', 'baseline-dINT']; [] = raw (no covariates).
# Guided by the collinearity check above: stage-1 change is ~orthogonal to S-A but strongly
# collinear with the baselines (baseline-dINT r~-0.83 => over-control), so S-A-only is the clean
# control; in stage 2 S-A / baseline-INT are modest and defensible.
# STAGE1_COVARS = ['S-A']
# STAGE2_COVARS = ['S-A']
# STAGE1_COVARS = ['baseline-INT']
# STAGE2_COVARS = ['baseline-INT']
STAGE1_COVARS = []
STAGE2_COVARS = []
# ==============================================================================

def _build_cov(stage, names):
    if not names:
        return None
    cols = [covariate_maps[stage][n] for n in names]
    return np.column_stack(cols) if len(cols) > 1 else cols[0]

cov_s1 = _build_cov('stage1', STAGE1_COVARS)
cov_s2 = _build_cov('stage2', STAGE2_COVARS)

# Gene / PET / cell-type spatial-correlation families run at Schaefer-400 only. At other resolutions
# they are skipped (results stay None) -- only the Yeo-7 enrichment and the myelin section are run.
res_gene_s1 = res_gene_s2 = None
res_pet_s1  = res_pet_s2  = None
res_cell_s1 = res_cell_s2 = None
# Baum myelin family runs at every resolution the maps are provided at (not gated to Schaefer-400).
res_baum_s1 = res_baum_s2 = None

if n_regions == 400:
    # Each family runs only if still present: None => skipped at this resolution or excluded
    # via the INCLUDE_*_MAPS toggles in the config cell.
    if df_genes is not None:
        res_gene_s1 = correlate_family(stage1_map, df_genes, cov_s1, distance_matrix, N_PERMS, 'genes P1')
        res_gene_s2 = correlate_family(stage2_map, df_genes, cov_s2, distance_matrix, N_PERMS, 'genes P2')
    if df_pet is not None:
        pet_cols = individual_cols + ['PET-PC1']   # individual PET markers + dominant PET axis (composites not plotted)
        res_pet_s1 = correlate_family(stage1_map, df_pet[pet_cols], cov_s1, distance_matrix, N_PERMS, 'PET P1')
        res_pet_s2 = correlate_family(stage2_map, df_pet[pet_cols], cov_s2, distance_matrix, N_PERMS, 'PET P2')
    if df_cells is not None:
        res_cell_s1 = correlate_family(stage1_map, df_cells, cov_s1, distance_matrix, N_PERMS, 'cells P1')
        res_cell_s2 = correlate_family(stage2_map, df_cells, cov_s2, distance_matrix, N_PERMS, 'cells P2')

# Baum myelin maps: separate family, available beyond Schaefer-400. Precomputed nulls on disk are
# keyed by the 'baum-*' column names, so generate_surrogates (inside correlate_family) reuses them.
if df_baum is not None:
    res_baum_s1 = correlate_family(stage1_map, df_baum, cov_s1, distance_matrix, N_PERMS, 'baum P1')
    res_baum_s2 = correlate_family(stage2_map, df_baum, cov_s2, distance_matrix, N_PERMS, 'baum P2')

_cols = ['r', 'p_smash', 'fdr_sig']
pd.set_option('display.float_format', lambda v: f'{v:.3f}')
for _name, _res, _cov in [('GENES — stage 1', res_gene_s1, STAGE1_COVARS),
                          ('PET — stage 1',   res_pet_s1,  STAGE1_COVARS),
                          ('CELLS — stage 1', res_cell_s1, STAGE1_COVARS),
                          ('BAUM — stage 1',  res_baum_s1, STAGE1_COVARS),
                          ('GENES — stage 2', res_gene_s2, STAGE2_COVARS),
                          ('PET — stage 2',   res_pet_s2,  STAGE2_COVARS),
                          ('CELLS — stage 2', res_cell_s2, STAGE2_COVARS),
                          ('BAUM — stage 2',  res_baum_s2, STAGE2_COVARS)]:
    if _res is None:
        continue
    print(f'=== {_name}  (control: {"+".join(_cov) if _cov else "raw"}; FDR within family) ===')
    print(_res[_cols].to_string())
    print()

if n_regions != 400:
    print(f'n_regions={n_regions} != 400; skipped gene/PET/cell spatial-correlation families '
          f'(only Yeo-7 enrichment + Baum myelin + myelin section are computed at this resolution).')


## Yeo-7 network enrichment

For each Yeo-7 network we take the mean stage value across its parcels and test it against a BrainSMASH surrogate null: the *stage map* is surrogated (preserving spatial autocorrelation), network means are recomputed for every surrogate, and a two-tailed deviation p-value compares the observed network mean to the null band. FDR is applied across the 7 networks. Both stages are shown, with per-network means and 95% null intervals in Yeo colors.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
from src.nulls import yeo_network_enrichment

yeo_systems = np.asarray(notebook_data['supporting_data']['yeo_systems'])

# Surrogate the stage maps (cached); compute enrichment and FDR across networks (both stages).
surr_stage1 = generate_surrogates(stage1_map, distance_matrix, 'stage1', n_perms=N_PERMS)
surr_stage2 = generate_surrogates(stage2_map, distance_matrix, 'stage2', n_perms=N_PERMS)
res_yeo_s1 = yeo_network_enrichment(stage1_map, yeo_systems, surr_stage1)
res_yeo_s2 = yeo_network_enrichment(stage2_map, yeo_systems, surr_stage2)
for _r in (res_yeo_s1, res_yeo_s2):
    _r['p_fdr_sig'] = multipletests(_r['p_smash'].values, alpha=0.05, method='fdr_bh')[0]

pd.set_option('display.float_format', lambda v: f'{v:.3f}')
for _name, _res in [('STAGE 1', res_yeo_s1), ('STAGE 2', res_yeo_s2)]:
    print(f'=== Yeo-7 enrichment — {_name} (two-tailed; FDR across {len(_res)} networks) ===')
    print(_res[['mean', 'null_lo', 'null_hi', 'p_smash', 'p_fdr_sig']].to_string())
    print()


# Stage 2: does subject-level DMN expansion predict cognition or SES?

Exploratory test of whether the stage-2 phenomenon (association-cortex / Default-network expansion of task-evoked INT modulation) carries individual-difference signal for behaviour. For each subject we compute a **DMN-selective deflection** score — mean |ΔINT| over Yeo-7 Default-network parcels minus mean |ΔINT| over all Schaefer-400 parcels — and ask, with two pre-specified directional tests, whether it predicts (i) NIH Toolbox cognition and (ii) family SES (log income-to-needs), each controlling for age (linear + quadratic, matching the GAM age form), sex, and in-scanner motion (mean FD). Behavioural variables are loaded and operationalized exactly as in the backup notebook.

Both hypotheses are directional-positive (more DMN expansion → higher cognition / SES). The two outcomes are analyzed independently and **two tests are run in total**; p-values are reported as-is, without multiple-comparison correction, and without scanning alternative measures, SES operationalizations, or age windows. This is exploratory — a null is an acceptable, reportable outcome and the analysis is built to be interpretable either way.

In [ ]:
import numpy as np
import pandas as pd

# Subject-level DMN-selective deflection: mean |ΔINT| over Yeo-7 Default-network parcels minus
# mean |ΔINT| over all Schaefer-400 parcels. Reuses the per-subject |ΔINT| (defl_reg, aligned to
# df_behavior row order) and the Yeo-7 assignment already in the notebook. One value per subject.
yeo_systems = np.asarray(notebook_data['supporting_data']['yeo_systems'])
dmn_mask    = yeo_systems == 'Default'

dmn_within = np.nanmean(defl_reg[:, dmn_mask], axis=1)
global_avg = np.nanmean(defl_reg, axis=1)
dmn_selective = dmn_within - global_avg

dmn_selective = pd.Series(dmn_selective, index=df_behavior.index, name='dmn_selective')
n_valid = int(np.isfinite(dmn_selective.values).sum())
print(f'DMN-selective deflection: {int(dmn_mask.sum())}/{dmn_mask.size} parcels in Yeo-7 Default; '
      f'N subjects with a valid value = {n_valid}')
print('Distribution:')
print(dmn_selective.describe().to_string())
print(f'  positive (DMN > whole-brain) in {int((dmn_selective > 0).sum())}/{n_valid} subjects')

In [ ]:
import numpy as np
import pandas as pd

# Behavioural variables, loaded/operationalized exactly as in the backup notebook.
beh_dir = timescales_processor.base_dir / 'behavior'

# SES — family income-to-needs ratio from socdem01.txt
df_socdem = pd.read_csv(
    beh_dir / 'socdem01.txt', delimiter='\t', skiprows=[1],
    usecols=['src_subject_id', 'annual_fam_inc', 'household_number_in_house'],
)
df_socdem['subject_id'] = df_socdem['src_subject_id'].astype(str) + 'V1MR'
df_socdem = df_socdem.drop(columns='src_subject_id').set_index('subject_id')
df_socdem['annual_fam_inc'] = df_socdem['annual_fam_inc'].replace(-999999, np.nan)
df_socdem.loc[df_socdem['annual_fam_inc'] == 0, 'annual_fam_inc'] = np.nan
df_socdem.loc[df_socdem['household_number_in_house'] < 1,  'household_number_in_house'] = np.nan
df_socdem.loc[df_socdem['household_number_in_house'] > 12, 'household_number_in_house'] = np.nan
HHS_BASE_2018, HHS_INCREMENT_2018 = 12_140, 4_320
df_socdem['poverty_line']    = HHS_BASE_2018 + HHS_INCREMENT_2018 * (df_socdem['household_number_in_house'] - 1)
df_socdem['income_to_needs'] = df_socdem['annual_fam_inc'] / df_socdem['poverty_line']
df_socdem['log_itn']         = np.log(df_socdem['income_to_needs'])

# Cognition — NIH Toolbox total cognitive composite (age-adjusted), crystallized fallback
df_cog = pd.read_csv(
    beh_dir / 'cogcomp01.txt', delimiter='\t', skiprows=[1],
    usecols=['src_subject_id', 'nih_totalcogcomp_ageadjusted', 'nih_crycogcomp_ageadjusted'],
)
df_cog['subject_id'] = df_cog['src_subject_id'].astype(str) + 'V1MR'
df_cog = (df_cog.drop(columns='src_subject_id').set_index('subject_id')
                .apply(pd.to_numeric, errors='coerce'))
df_cog['cognition'] = df_cog['nih_totalcogcomp_ageadjusted'].fillna(df_cog['nih_crycogcomp_ageadjusted'])

# Per-subject analysis frame: DMN-selective measure + covariates, then join behaviour by ID.
# Covariates: age, age^2 (matching the GAM nonlinear age form), sex, mean FD.
df_dmn = pd.DataFrame({
    'dmn_selective': dmn_selective,
    'age':           df_behavior['age'].values,
    'sex':           df_behavior['sex'].values,
    'meanFD':        df_behavior['meanFD'].values,
}, index=df_behavior.index)
df_dmn['age2'] = df_dmn['age'] ** 2
df_dmn = df_dmn[np.isfinite(df_dmn['dmn_selective'])]   # subjects with a valid DMN measure

df_dmn = df_dmn.join(df_socdem[['log_itn']], how='left').join(df_cog[['cognition']], how='left')

covars = ['age', 'age2', 'sex', 'meanFD']
n_cog = int(df_dmn.dropna(subset=['cognition'] + covars + ['dmn_selective']).shape[0])
n_ses = int(df_dmn.dropna(subset=['log_itn']  + covars + ['dmn_selective']).shape[0])
print(f'Subjects with valid DMN-selective measure: {len(df_dmn)}')
print(f'  matched N — cognition (NIH Toolbox): {n_cog}')
print(f'  matched N — SES (log income-to-needs): {n_ses}')

In [ ]:
import pandas as pd
from snaplab_tools.stats import partial_corr_controlled

# Two pre-specified positive-direction partial correlations: Stage-2 DMN-selective deflection ->
# behaviour, controlling for age, age^2, sex, meanFD. partial_corr_controlled lives in
# snaplab_tools.stats (shared).
covars = ['age', 'age2', 'sex', 'meanFD']
tests = [
    ('Cognition (NIH Toolbox)', 'cognition'),
    ('SES (log income-to-needs)', 'log_itn'),
]
results_dmn = {label: partial_corr_controlled(df_dmn, 'dmn_selective', outcome, covars)
               for label, outcome in tests}

summary_dmn = pd.DataFrame([
    {'outcome': label, 'r': res['r'], 'p_two_tailed': res['p_two'],
     'p_one_tailed_pos': res['p_one_pos'], 'N': res['n'],
     'covariates': 'age + age^2 + sex + meanFD'}
    for label, res in results_dmn.items()
]).set_index('outcome')

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
pd.set_option('display.max_columns', None, 'display.width', 200)
print('Stage-2 DMN-selective deflection -> behaviour (partial correlations)')
print('Two pre-specified directional (positive) tests, run independently; p-values reported '
      'as-is (no multiple-comparison correction).')
print('=' * 92)
print(summary_dmn.to_string())
print('=' * 92)
for label, res in results_dmn.items():
    direction = 'matches' if res['r'] > 0 else 'OPPOSITE to'
    verdict = ('NULL' if res['p_two'] >= 0.05 else
               ('supports hypothesis' if res['r'] > 0 else 'significant but wrong direction'))
    print(f'{label}: partial r = {res["r"]:+.3f}, p(two-tailed) = {res["p_two"]:.3f}, '
          f'N = {res["n"]} — sign {direction} the positive hypothesis -> {verdict}')

# Myelin analysis: Baum maps

In [ ]:
df_baum

In [ ]:
baum_dir = '/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales/data/supporting_data/baum_data'
# baum_metrics = ['baum-aroc', 'baum-aoms', 'baum-deriv2', 'baum-young', 'baum-mid', 'baum-old']
# Empty when the Baum myelin family is excluded (INCLUDE_MYELIN_MAPS=False) or unavailable,
# so the per-metric loop below is simply skipped.
baum_metrics = ['baum-aroc', 'baum-deriv2'] if df_baum is not None else []
if df_baum is None:
    print('Baum myelin family not present (INCLUDE_MYELIN_MAPS=False or unavailable); '
          'skipping per-metric myelin section.')

for baum_metric in baum_metrics:
    baum_data = df_baum[baum_metric]
    
    surr = generate_surrogates(baum_data, distance_matrix, baum_metric, n_perms=N_PERMS)
    results = corr_with_covar_null(stage2_map, baum_data, cov_s2, surr)
    print(results)

    _vmax = float(np.nanmax(np.abs(baum_data)))
    _vmin = float(np.nanmin(np.abs(baum_data)))

    _fig, _ = plot_brain_surface_data(data_vector=baum_data, parcellation=f'schaefer_{n_regions}-7', cmap='magma',
                                    vmin=_vmin, vmax=_vmax, cbar_label=baum_metric)
    surface_paths[_lab] = os.path.join(nas_scratch_dir, f'figure_5_surface_{baum_metric}.png')
    _fig.savefig(surface_paths[_lab], dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(5, 2.5), dpi=300)
    plot_correlation(pd.Series(stage1_map, name='Stage 1'), pd.Series(baum_data, name=baum_metric),
                    axes[0]
                    , method='pearson', show_stats=True, show_colorbar=False, fontsize=7)
    plot_correlation(pd.Series(stage2_map, name='Stage 2'), pd.Series(baum_data, name=baum_metric),
                    axes[1], method='pearson', show_stats=True, show_colorbar=False, fontsize=7)
    fig.tight_layout()
    plt.show()

# Implement send-receive analysis here

In [ ]:
print(notebook_data.keys())
print(notebook_data['adjacency_matrices'].shape)

In [ ]:
# ── Exploratory: within-DMN send/receive communication asymmetry vs. the Stage-2 map ──
# Relates the group-mean directional communication asymmetry of the structural connectome to
# the Stage-2 developmental effect (GAM-summarized net change in |ΔINT| after age 17), within
# the Yeo-7 Default-mode network. EXPLORATORY: two-sided, no directional hypothesis; a null is
# uninformative here, not evidence of absence (see caveats printed below).
#
# Sign convention (from src.communication_asymmetry): each measure is oriented "higher = more
# efficient" BEFORE the send/receive reduction, so asymmetry = send - receive > 0 means the
# parcel is BROADCASTER-leaning (sends more efficiently than it receives).
import os
import numpy as np
import scipy as sp
import pandas as pd
import src.communication_asymmetry as comm_asym

# ── Inputs already in the namespace (see earlier cells) ───────────────────────────────────
#   notebook_data['adjacency_matrices']         -> (400, 400, n_sub) structural streamline stack
#   D_euclid (below) -> (400, 400) Euclidean centroid distances; used as the PHYSICAL distance
#     for the communication models only (not for the BrainSMASH null, which is geodesic).
#   notebook_data['subject_ids']                -> subject IDs aligned to axis 2 of the stack
#   stage2_map                                  -> (400,) Stage-2 net-change map
#   yeo_systems                                 -> (400,) Yeo-7 labels ('Default' = DMN)
W_stack     = notebook_data['adjacency_matrices']
D_euclid    = notebook_data['supporting_data']['distance_matrix']
sr_subject_ids = list(notebook_data['subject_ids'])

# ── Step 1: three-measure send/receive battery over all subjects (asymmetry component only) ──
# run_over_subjects returns (long_df, stacked, summary); stacked[measure]['asymmetry'] is
# (n_sub, 400), row-aligned to sr_subject_ids. Weights are passed un-normalized (streamline
# counts); the module leaves subject-level normalization to the caller, and we keep raw counts
# for this first pass. Disconnected slices are restricted to their largest component (NaN for
# dropped parcels) under the default 'warn' policy.
#
# CACHING: this battery is slow (three graph algorithms per subject). We cache the reduced
# per-subject send/receive/asymmetry to disk, content-addressed by the subject's connectome
# slice + D + policy, so it is computed once and reloaded thereafter. Adding/removing subjects
# only recomputes the new ones; regenerating the connectome invalidates stale entries
# automatically. Delete the directory to force a full recompute.
sr_cache_dir = os.path.join(timescales_processor.output_dir, 'results', dataset, 'comm_asym')
_, sr_stacked, sr_summary = comm_asym.run_over_subjects(
    W_stack, D_euclid, sr_subject_ids, connected_policy='warn', cache_dir=sr_cache_dir)
print('Send/receive battery:', sr_summary['n_subjects'], 'subjects;',
      sr_summary['n_disconnected_subjects'], 'triggered the disconnected-component path.')
print('Cache:', sr_cache_dir, '->', len(os.listdir(sr_cache_dir)), 'subject files.')

sr_measures = ['navigation_efficiency', 'search_information', 'diffusion_efficiency']

In [ ]:
# ── Step 2: group mean asymmetry per parcel (nanmean over subjects) -> 3 length-400 maps ──
sr_group_asym = {m: np.nanmean(sr_stacked[m]['asymmetry'], axis=0) for m in sr_measures}

# ── Step 3: DMN mask; restrict group maps and the Stage-2 map to DMN parcels ──
# Use a single common parcel set (DMN parcels finite in the Stage-2 map AND all three asymmetry
# maps) so the BrainSMASH surrogates and all three correlations share the same n_DMN.
dmn_mask = np.asarray(yeo_systems) == 'Default'
sr_finite = dmn_mask & np.isfinite(stage2_map)
for m in sr_measures:
    sr_finite &= np.isfinite(sr_group_asym[m])
dmn_idx = np.where(sr_finite)[0]
n_dmn = dmn_idx.size

stage2_dmn = stage2_map[dmn_idx]

# ── Step 5 (surrogates first, shared across measures): BrainSMASH nulls of the Stage-2 DMN map ──
# Per-hemisphere GEODESIC basis via src.nulls: build a full-length map that is NaN outside the DMN
# so each DMN parcel is surrogated within its own hemisphere's geodesic block, then keep the DMN
# columns. This upgrades the earlier Euclidean approximation to the geodesic basis recommended for
# BrainSMASH on surface parcellations.
N_SR_SURROGATES = 5000
_stage2_dmn_full = np.full(stage2_map.shape[0], np.nan)
_stage2_dmn_full[dmn_idx] = stage2_dmn
_surr_full = nulls.generate_surrogates(
    _stage2_dmn_full, name='stage2_dmn', n_regions=n_regions, n_perms=N_SR_SURROGATES,
    kind='geodesic', distance_matrix=distance_matrix, hemi=hemi_labels,
    cache_dir=surrogate_cache_dir, seed=0)
stage2_dmn_surrogates = _surr_full[:, dmn_idx]  # (N_SR_SURROGATES, n_dmn)

# ── Steps 4 & 5: observed corr + two-sided BrainSMASH p per measure ──
def _pearson_smash_p(asym_dmn, target, surrogates):
    """Observed corr(asym_dmn, target) and two-sided p from the surrogate null of target."""
    corr_obs = sp.stats.pearsonr(asym_dmn, target)[0]
    null = np.array([sp.stats.pearsonr(asym_dmn, surrogates[k])[0]
                     for k in range(surrogates.shape[0])])
    # Two-sided: fraction of surrogate |corr| at least as extreme as observed (+1 smoothing).
    p_two = (1.0 + np.sum(np.abs(null) >= np.abs(corr_obs))) / (1.0 + null.size)
    return corr_obs, p_two

sr_rows = []
for m in sr_measures:
    corr, p_smash = _pearson_smash_p(sr_group_asym[m][dmn_idx], stage2_dmn, stage2_dmn_surrogates)
    sr_rows.append({'measure': m, 'corr': corr, 'p_smash': p_smash, 'n_DMN_parcels': n_dmn})

# ── Step 6: tidy summary ──
sr_results = pd.DataFrame(sr_rows)
print('\nWithin-DMN group-mean communication asymmetry (send - receive) vs. Stage-2 map')
print('asymmetry > 0 = broadcaster-leaning (send > receive); two-sided BrainSMASH p:')
print(sr_results.to_string(index=False))

# ── Reporting caveats ──
print(f'\nCaveats: n = {n_dmn} DMN parcels only -> a low-powered spatial test; a non-significant')
print('result is uninformative (not evidence of absence). Exploratory and two-sided: any sign')
print('is post hoc, not predicted. BrainSMASH null uses a per-hemisphere geodesic distance basis.')


In [ ]:
_map = sr_group_asym['navigation_efficiency']
fig, axes = plt.subplots(1, 1, figsize=(2.5, 2.5), dpi=300)
plot_correlation(pd.Series(stage2_dmn, name='Stage 2 (DMN)'), pd.Series(_map[dmn_idx], name='Comm. Asymmetry (DMN)'),
                axes, method='pearson', show_stats=True, show_colorbar=False, fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
for yeo_system in np.unique(yeo_systems):
    
    mask = np.asarray(yeo_systems) == yeo_system
    sr_finite = mask & np.isfinite(stage2_map)
    for m in sr_measures:
        sr_finite &= np.isfinite(sr_group_asym[m])
    mask_idx = np.where(sr_finite)[0]
    stage2_system = stage2_map[mask_idx]
    
    print(yeo_system, sp.stats.pearsonr(stage2_system, _map[mask_idx]))

# Figure 5 final

Two-column developmental-stage layout (**Stage 1** left, **Stage 2** right):

- **Top row** — Yeo-7 network enrichment: per-network mean net Δ|INT deflection| against its BrainSMASH surrogate null (stars = BH-FDR significant across the 7 networks; DMN annotated).
- **Middle row** — only the BH-FDR-significant molecular/myelin brain correlations, each observed partial *r* shown against its faithful BrainSMASH null violin.

Reuses `res_yeo_s1/s2` (enrichment) and `res_{gene,pet,cell,baum}_s1/s2` (associations) from the cells above — run those first.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════════════════
# FIGURE 5 FINAL — 2-column developmental-stage layout (Stage 1 | Stage 2)
#   Row 1 (top):    Yeo-7 network enrichment — per-network mean net Δ|INT deflection| vs its
#                   BrainSMASH surrogate null (stars = BH-FDR significant across the 7 networks).
#   Row 2 (middle): ONLY the BH-FDR-significant molecular/myelin brain correlations, each shown
#                   as its observed partial r against the faithful BrainSMASH null violin. Stage 2
#                   has few sig corrs, so its violin is narrowed and the freed space carries the
#                   Stage-2 DMN communication-asymmetry scatter (from the send/receive cells above).
# All nulls/observations are reused from the association (res_*_s1/s2), enrichment (res_yeo_s1/s2),
# and send/receive (sr_group_asym, dmn_idx, stage2_dmn) cells above — run those first.
# Self-contained: redefines its own helpers so it renders independently of the OLD assembly cell.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from snaplab_tools.plotting.plotting import YEO7_COLORS, plot_correlation

# Communication-asymmetry measure shown in the Stage-2 scatter (matches the exploratory cell above).
SR_SCATTER_MEASURE = 'navigation_efficiency'

# Family point colours (shared with the OLD assembly for visual continuity).
GENE_COL, PET_COL, CELL_COL, BAUM_COL = '#55A868', '#DD8452', '#8172B3', '#4C72B0'


def _violin_panel(ax, labels, nulls, obs, sig, point_colors, ylabel, hline=0.0,
                  rotation=45, label_fs=6, show_stars=True):
    """Grey BrainSMASH null violins with observed points on top. Non-significant entries
    (both violin body and point) are drawn more transparent; significant points are starred
    when show_stars."""
    xpos = np.arange(len(labels))
    clean = [np.asarray(n, float)[np.isfinite(np.asarray(n, float))] for n in nulls]
    parts = ax.violinplot(clean, positions=xpos, widths=0.8, showextrema=False)
    for pc, s in zip(parts['bodies'], sig):
        pc.set_facecolor('0.7'); pc.set_alpha(0.45 if s else 0.15)
        pc.set_edgecolor('0.45'); pc.set_linewidth(0.4)
    fc = [(*mcolors.to_rgb(c), 1.0 if s else 0.15) for c, s in zip(point_colors, sig)]
    ax.scatter(xpos, obs, c=fc, s=34, zorder=3, edgecolor='k', linewidth=0.4)
    if show_stars:
        for _x, _o, _s in zip(xpos, obs, sig):
            if _s:
                ax.annotate('*', (_x, _o), textcoords='offset points', xytext=(0, 2),
                            ha='center', fontsize=10, zorder=4)
    ax.axhline(hline, color='gray', ls='--', lw=0.8, alpha=0.7, zorder=0)
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels, rotation=rotation, ha='right' if rotation else 'center', fontsize=label_fs)
    ax.set_ylabel(ylabel, fontsize=7); ax.tick_params(labelsize=6, pad=1)
    ax.spines[['top', 'right']].set_visible(False)


def _enrichment_panel(ax, res_yeo):
    """Row 1: Yeo-7 network enrichment violins for one stage."""
    nets = list(res_yeo.index)
    _violin_panel(ax, nets,
                  [res_yeo.loc[n, 'null'] for n in nets],
                  [res_yeo.loc[n, 'mean'] for n in nets],
                  [bool(res_yeo.loc[n, 'p_fdr_sig']) for n in nets],
                  [YEO7_COLORS.get(n, (0.5, 0.5, 0.5)) for n in nets],
                  ylabel='Δ|INT deflection|',
                  hline=float(np.nanmean(res_yeo['null_mean'].astype(float))),
                  rotation=0, label_fs=6)


def _sig_corr_panel(ax, fam_results):
    """Row 2: ONLY the FDR-significant molecular/myelin correlations, vs their BrainSMASH nulls.
    fam_results: list of (label, kind, colour, results_df_or_None)."""
    sig_items, kinds = [], set()  # sig_items: (name, colour, null_vec, r)
    for _lab, _kind, _col, _res in fam_results:
        if _res is None:
            continue
        for n in _res.index:
            if bool(_res.loc[n, 'fdr_sig']):
                sig_items.append((n, _col, _res.loc[n, 'null'], float(_res.loc[n, 'r'])))
                kinds.add(_kind)
    if not sig_items:
        ax.axis('off')
        ax.text(0.5, 0.5, 'no FDR-significant\ncorrelations', ha='center', va='center',
                fontsize=8, color='0.5', transform=ax.transAxes)
        return
    _violin_panel(ax, [n for n, _, _, _ in sig_items],
                  [nl for _, _, nl, _ in sig_items],
                  [r for _, _, _, r in sig_items],
                  [True] * len(sig_items),
                  [c for _, c, _, _ in sig_items],
                  ylabel=r'Correlation ($r$)', hline=0.0, rotation=0,
                  label_fs=6, show_stars=False)
    handles = [Patch(color=col, label=lab) for lab, kind, col in
               [('Gene', 'gene', GENE_COL), ('PET', 'pet', PET_COL),
                ('Cell', 'cell', CELL_COL), ('Baum', 'baum', BAUM_COL)] if kind in kinds]
    ax.legend(handles=handles, fontsize=6, loc='best', frameon=False)


def _comm_asym_scatter(ax, measure=SR_SCATTER_MEASURE):
    """Stage-2 (DMN) developmental change vs group-mean communication asymmetry over DMN parcels.
    Reproduces the exploratory send/receive scatter (Pearson r + stats drawn by plot_correlation)."""
    y = sr_group_asym[measure][dmn_idx]
    plot_correlation(pd.Series(stage2_dmn, name='Stage 2 (DMN)'),
                     pd.Series(y, name='Comm. asym. (DMN)'),
                     ax, method='pearson', show_stats=True, show_colorbar=False, fontsize=6)
    ax.tick_params(labelsize=6, pad=1)


# ── Per-stage config: enrichment result + the four correlation families ─────────────────────
stage_cfgs = [
    dict(name='Stage 1', yeo=res_yeo_s1,
         fams=[('Genes', 'gene', GENE_COL, res_gene_s1), ('PET', 'pet', PET_COL, res_pet_s1),
               ('Cells', 'cell', CELL_COL, res_cell_s1), ('Baum', 'baum', BAUM_COL, res_baum_s1)]),
    dict(name='Stage 2', yeo=res_yeo_s2,
         fams=[('Genes', 'gene', GENE_COL, res_gene_s2), ('PET', 'pet', PET_COL, res_pet_s2),
               ('Cells', 'cell', CELL_COL, res_cell_s2), ('Baum', 'baum', BAUM_COL, res_baum_s2)]),
]

# ── 2 columns (stages) x 2 rows (enrichment, FDR-sig brain corrs) ───────────────────────────
_row_labels = ['Yeo-7 enrichment', 'FDR-sig brain corrs']
_letters = [['a', 'b'], ['c', 'd']]

fig = plt.figure(figsize=(8, 5), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], wspace=0.25, hspace=0.4)

for _col, cfg in enumerate(stage_cfgs):
    # Row 1 — enrichment
    ax0 = fig.add_subplot(gs[0, _col])
    _enrichment_panel(ax0, cfg['yeo'])
    # ax0.set_title(cfg['name'], loc='center', fontsize=12, fontweight='bold', pad=16)
    # ax0.set_title(_letters[0][_col], loc='left', fontsize=11, fontweight='bold', pad=16)

    # Row 2 — FDR-significant brain correlations. Stage 1 (col 0) keeps a full-width violin panel;
    # Stage 2 (col 1) has few sig corrs, so its violin is narrowed and the freed half carries the
    # Stage-2 DMN communication-asymmetry scatter.
    if _col == 0:
        ax1 = fig.add_subplot(gs[1, _col])
        _sig_corr_panel(ax1, cfg['fams'])
        # ax1.set_title(_letters[1][_col], loc='left', fontsize=11, fontweight='bold', pad=10)
    else:
        _sub = gs[1, _col].subgridspec(1, 2, width_ratios=[1.0, 1.3], wspace=0.55)
        ax1 = fig.add_subplot(_sub[0, 0])
        _sig_corr_panel(ax1, cfg['fams'])
        # ax1.set_title('d', loc='left', fontsize=11, fontweight='bold', pad=10)
        ax_sc = fig.add_subplot(_sub[0, 1])
        _comm_asym_scatter(ax_sc)
        # ax_sc.set_title('e', loc='left', fontsize=11, fontweight='bold', pad=10)

    # Row descriptors down the left edge (drawn once, on the left column).
    # if _col == 0:
    #     for _r, _ax in enumerate((ax0, ax1)):
    #         _ax.annotate(_row_labels[_r], xy=(-0.225, 0.5), xycoords='axes fraction',
    #                      rotation=90, va='center', ha='center', fontsize=9, fontweight='bold')

_out = os.path.join(nas_scratch_dir, 'figure_5_final.svg')
fig.savefig(_out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved ->', _out)
